### Tutorial FMUs in BuilDyn

BuilDyn provides the user with a custom FMU object. This is a wrapper around the FmPy FMUSlave object, which is hard to handle and is missing core features.

To begin, we start by initializing a FMU object. We use the FMU from BuilDa 2.0 for demonstration, but in reality every FMU can be used.

In [ ]:
from buildyn import FMU

# Example FMU from BuilDa 2.0
fmu_path = "resources/building_dymola_linux_v3.fmu"

# Initial variables from the BuilDa 2.0 FMU - without those the FMU cannot be initialized.
start_variables = {
    "weaDat.filNam": "resources/Munich.mos",
    "internalGain.fileName": "resources/NoActivity.txt",
    "hygienicalWindowOpening.fileName": "resources/no_opening.txt",
    "UseInternalController.k": 0
}

# FMU object from the buildyn package
fmu = FMU(fmu_file=fmu_path, init_values=start_variables)

For further experiments, we first define a function that transforms the timestamps of the buildings FMU into real time, so we can better observer the times.

In [ ]:
import pandas as pd
from copy import deepcopy

def transform_timestamp(df: pd.DataFrame, column_name: str = "time", start: pd.Timestamp = pd.Timestamp("2017-01-01")):

    df["timestamp"] = start + pd.to_timedelta(df[column_name], unit="s")


def plot_output(df: pd.DataFrame, variable: list[str] | str ):

    df_ = deepcopy(df)
    transform_timestamp(df=df_)
    df_ = df_.set_index("timestamp")

    df_[variable].plot()


By defining observables, we can retrieve timeseries data directly from the FMU. In the following code cell we simulate the FMU for 96 timesteps 15min each, which would be 1 day.   
The duration of 1 timestep and the total time for simulation can obviously be changed.

In [ ]:
# For the toy example, we choose the temperature inside the building (thermalZone.TAir) in Kelvin and the signal for the heating (ctrSignalHeating) as observable variables.
observables = ["thermalZone.TAir", "ctrSignalHeating"]

# We can simulate the FMU simply by calling the simulate method.
one_day_df = fmu.simulate(observables=observables)

plot_output(one_day_df, observables)

In this FMU we can also dynamically set variables. We demonstrate it for another FMU that is a copy of the original one.

In [ ]:
# Make a copy of the original FMU, so the original one does not get overridden. A new object is created here with the configuration of the old FMU.
fmu_copy = fmu.__copy__()

# Here we set the heating to 1, meaning full heating power over the whole time horizon. This obviously changes the indoor temperature.
fmu_copy.set_variable("ctrSignalHeating", 1)

fmu_copy_output = fmu_copy.simulate(observables=observables)

plot_output(fmu_copy_output, observables)

**Current limitation note:** The FMU reset function does not work. That means, you can only simulate a FMU once and then it shows undefined behavior. To overcome this problem until the bug is fixed, copy the fmu and call simulate on the copied version. This works just as fine.